# Load and process Bacillus data

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO, Phylo, AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load `.nwk` tree

In [ ]:
clonal_tree = Phylo.read("../data/bacillus/bacillus.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

In [ ]:
edge = np.loadtxt("../data/bacillus/clonal_edge.csv", delimiter=",", dtype=float)
node_height = np.loadtxt("../data/bacillus/clonal_node_height.csv", delimiter=",", dtype=float)

In [ ]:
edge, node_height

In [ ]:
node_height = node_height[~np.isnan(node_height)]
node_height

In [ ]:
# np.savetxt("../data/bacillus/clonal_edge.csv", edge, delimiter=",")
# np.savetxt("../data/bacillus/clonal_node_height.csv", node_height, delimiter=",")
# clonal_edge = np.loadtxt("../data/bacillus/clonal_edge.csv", delimiter=",", dtype=float)
# clonal_node_height = np.loadtxt("../data/bacillus/clonal_node_height.csv", delimiter=",", dtype=float)

## Load `.xmfa` data

In [ ]:
def read_xmfa_biopython(file_path):
    blocks = list(AlignIO.parse(file_path, "mauve"))
    return blocks

In [ ]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing a '-' completely removed.
    """
    align_len = alignment_block.get_alignment_length()
    
    valid_columns = [i for i in range(align_len) if "-" not in alignment_block[:, i]]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    return MultipleSeqAlignment(cleaned_records)

In [ ]:
xmfa_path = "../data/bacillus/bacillus.xmfa"
aligned_blocks = read_xmfa_biopython(xmfa_path)

print(f"Total blocks successfully loaded: {len(aligned_blocks)}")

### Create info table

In [ ]:
block_info_df = pd.DataFrame({'Gene_ID': range(len(aligned_blocks))})
block_info_df['Gene_Length'] = None
block_info_df['Start_pos'] = None
block_info_df['End_pos'] = None
block_info_df['Alignment'] = True
block_info_df

### Create summary stats table + count segregating sites

In [ ]:
block_summary = np.full((len(aligned_blocks), 46), np.nan)
block_summary

In [ ]:
total_S = 0
total_length = 0

In [ ]:
np.random.seed(100)
clonal_tree = ClonalTree(n=13)

clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

## Compute info table and summary stats

In [ ]:
def divide_evenly(n, parts):
    quotient, remainder = divmod(n, parts)
    
    # Create the list with the distributed remainder
    parts = [quotient + 1] * remainder + [quotient] * (parts - remainder)
    
    return parts

In [ ]:
gene_id = []
gene_length = []
start_pos = []
end_pos = []
summary_stats_list = []

for i in tqdm(range(len(aligned_blocks)), desc="Processing genes"):
    # get the raw block and remove gap columns
    raw_block = aligned_blocks[i]
    clear_block = remove_gap_columns(raw_block)

    # add the gene length and start/end positions to the info table
    if clear_block.get_alignment_length() <= 10000:
        gene_id.append(str(i))
        gene_length.append(clear_block.get_alignment_length())
        start_pos.append(raw_block[0].annotations.get('start'))
        end_pos.append(raw_block[0].annotations.get('start') + clear_block.get_alignment_length() - 1)

        # convert sequences to boolean matrix
        sequences = []
        for record in clear_block:
            seq_chars = list(str(record.seq).upper())
            sequences.append(seq_chars)

        char_matrix = np.array(sequences)
        reference_seq = char_matrix[0]
        bool_matrix = (char_matrix != reference_seq)

        # compute summary statistics for the boolean matrix
        summary_stats = segment_summary_stats(clonal_tree, bool_matrix)
        summary_stats_list.append(summary_stats)

        # compute segregating sites
        has_true = bool_matrix.any(axis=0)
        has_false = ~bool_matrix.all(axis=0)
        idx_seg = np.where(has_true & has_false)[0]
        total_S += idx_seg.size
        total_length += clear_block.get_alignment_length()
    else:
        sub_segments = int(clear_block.get_alignment_length() / 10000) + 1
        sub_length = divide_evenly(clear_block.get_alignment_length(), sub_segments)
        sub_start = raw_block[0].annotations.get('start')

        # convert sequences to boolean matrix
        sequences = []
        for record in clear_block:
            seq_chars = list(str(record.seq).upper())
            sequences.append(seq_chars)

        char_matrix = np.array(sequences)
        reference_seq = char_matrix[0]
        bool_matrix = (char_matrix != reference_seq)

        for j in range(sub_segments):
            gene_id.append(f"{i}_{j}")
            gene_length.append(sub_length[j])
            start_pos.append(sub_start)
            end_pos.append(sub_start + sub_length[j] - 1)
            sub_start += sub_length[j]

            # compute summary statistics for the boolean matrix
            sub_bool_matrix = bool_matrix[:, sum(sub_length[:j]):sum(sub_length[:j+1])]
            summary_stats = segment_summary_stats(clonal_tree, sub_bool_matrix)
            summary_stats_list.append(summary_stats)

            # compute segregating sites
            has_true = sub_bool_matrix.any(axis=0)
            has_false = ~sub_bool_matrix.all(axis=0)
            idx_seg = np.where(has_true & has_false)[0]
            total_S += idx_seg.size
            total_length += sub_length[j]

In [ ]:
block_info_df = pd.DataFrame({'Gene_ID': gene_id})
block_info_df['Gene_Length'] = gene_length
block_info_df['Start_pos'] = start_pos
block_info_df['End_pos'] = end_pos
block_info_df['Alignment'] = True
block_info_df

-----

In [ ]:
block_info_df.to_csv("../data/bacillus/bacillus_block_info.csv", index=False)

In [ ]:
np.savetxt("../data/bacillus/bacillus_block_summary_stats.csv", block_summary, delimiter=",")

In [ ]:
float(total_S / total_length / np.sum(edge[:, 2]) * 2)

0.047598670040356145